# 🚀 Alien Rescue — Game Analytics

**Analyzing 159 players · 85,194 log events · 1-hour gameplay session**

> *"What separates successful players from those who fail —  
> and how can the game be improved to help everyone succeed?"*

---

### Dataset
**Source:** Liu, S. & Liu, M. (2019). *Data on player activity and characteristics in a Serious Game Environment.*  
Data in Brief. DOI: [10.1016/j.dib.2019.104965](https://doi.org/10.1016/j.dib.2019.104965)

**Game:** Alien Rescue — players must find suitable planets for 6 displaced alien species using scientific tools.

| File | Rows | Description |
|---|---|---|
| Log_Raw | 85,194 | Every click, note, and navigation action |
| Consoles | 5,916 | Tool open/close events |
| Gates | 4,460 | Zone door crossings |
| Duration_Characteristics | 159 | Per-player summary + psychological scores |

---

### Analysis Structure

| # | Section | Business Question |
|---|---|---|
| 1 | Data Loading & Cleaning | How do we prepare 85k raw events? |
| 2 | Feature Engineering | How do we summarize each player in one row? |
| 3 | Exploratory Analysis | What does the data tell us at first glance? |
| 4 | Player Segmentation | Who are our players? Are they all the same? |
| 5 | Early Warning System | Can we detect struggling players in first 20 min? |
| 6 | Tool Analysis | Which tools create value — which need redesign? |


---
## 01 — Data Loading & Cleaning

### Why this matters
Raw game log data is messy by nature. Before any analysis, we need to:
- Standardize column names (original names had `|__dataLog__` prefixes)
- Parse timestamps from string to datetime format
- Remove duplicate player records
- Understand what each table represents

### What each file contains
- **Log_Raw**: every single player action — the richest source
- **Consoles**: which tools were opened/closed and when
- **Gates**: which game zones were entered and when  
- **Duration_Characteristics**: pre-computed per-player summaries + survey data (metacognition, goal orientation, solution score)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
OUTPUT_DIR = '../outputs/figures/'

log_raw = pd.read_csv('../data/Log_Raw.csv', sep='\t', on_bad_lines='skip')
consoles = pd.read_csv('../data/Consoles.csv', sep='\t', on_bad_lines='skip')
gates = pd.read_csv('../data/Gates.csv', sep='\t', on_bad_lines='skip')
duration = pd.read_csv('../data/Duration_Charateristics.csv')

print("log_raw shape:", log_raw.shape)
print("consoles shape:", consoles.shape)
print("gates shape:", gates.shape)
print("duration shape:", duration.shape)


In [ ]:
log_raw.columns = ['action', 'note', 'timestamp', 'tool', 'user_id']
consoles.columns = ['action', 'timestamp', 'tool', 'user_id']
gates.columns = ['action', 'note', 'timestamp', 'user_id']
duration.columns = [
    'user_id',
    'dur_alien_db', 'dur_comm_center', 'dur_concepts_db',
    'dur_mission_control', 'dur_missions_db', 'dur_notebook',
    'dur_periodic_table', 'dur_probe_design', 'dur_solar_db', 'dur_spectra',
    'gender', 'mc_average', 'solution_score',
    'tap', 'tav', 'sap', 'sav', 'oap', 'oav'
]

log_raw['timestamp'] = pd.to_datetime(log_raw['timestamp'], errors='coerce')
consoles['timestamp'] = pd.to_datetime(consoles['timestamp'], errors='coerce')
gates['timestamp'] = pd.to_datetime(gates['timestamp'], errors='coerce')

duration = duration.drop_duplicates(subset='user_id', keep='first')
duration = duration.dropna(subset=['solution_score'])

print("Unique players:", len(duration))
print("NaT timestamps in log_raw:", log_raw['timestamp'].isna().sum())


---
## 02 — Feature Engineering

### Why this matters
We have 85,194 rows — one per event. But to compare players and build segments,  
we need **one row per player** that summarizes their entire session.

This process is called **feature engineering**: transforming raw events into meaningful metrics.

### Features we'll create

| Feature | Source | What it captures |
|---|---|---|
| `total_actions` | log_raw | Overall engagement level |
| `notes_taken` | log_raw | Knowledge organization (metacognition proxy) |
| `section_clicks` | log_raw | Information-seeking behavior |
| `probe_actions` | log_raw | Problem-solving engagement |
| `total_gate_crossings` | gates | Map exploration |
| `unique_tools_used` | log_raw | Breadth of tool usage |
| `notebook_ratio` | derived | % of session time spent on notes |
| `action_density` | derived | Actions per minute (pace of play) |


In [ ]:
total_actions = log_raw.groupby('user_id')['action'].count()
total_actions.name = 'total_actions'

note_rows = log_raw[log_raw['action'].str.contains('Creat Note', na=False)]
notes_taken = note_rows.groupby('user_id')['action'].count()
notes_taken.name = 'notes_taken'

unique_tools = log_raw.groupby('user_id')['tool'].nunique()
unique_tools.name = 'unique_tools_used'

section_rows = log_raw[log_raw['action'] == 'Click Section']
section_clicks = section_rows.groupby('user_id')['action'].count()
section_clicks.name = 'section_clicks'

probe_rows = log_raw[log_raw['tool'].str.contains('Probe|probe', na=False)]
probe_actions = probe_rows.groupby('user_id')['action'].count()
probe_actions.name = 'probe_actions'

total_gates = gates.groupby('user_id')['action'].count()
total_gates.name = 'total_gate_crossings'

gate_pivot = gates.groupby(['user_id', 'note'])['action'].count().unstack(fill_value=0)
gate_pivot.columns = ['gate_' + c.lower().replace(' ', '_') for c in gate_pivot.columns]

open_rows = consoles[consoles['action'] == 'Open']
console_opens = open_rows.groupby('user_id')['action'].count()
console_opens.name = 'console_open_count'

unique_consoles = open_rows.groupby('user_id')['tool'].nunique()
unique_consoles.name = 'unique_consoles'

print("Features computed successfully")


In [ ]:
player_profile = duration.set_index('user_id')
player_profile = player_profile.join(total_actions, how='left')
player_profile = player_profile.join(notes_taken, how='left')
player_profile = player_profile.join(unique_tools, how='left')
player_profile = player_profile.join(section_clicks, how='left')
player_profile = player_profile.join(probe_actions, how='left')
player_profile = player_profile.join(total_gates, how='left')
player_profile = player_profile.join(gate_pivot, how='left')
player_profile = player_profile.join(console_opens, how='left')
player_profile = player_profile.join(unique_consoles, how='left')
player_profile = player_profile.reset_index()

cols_to_fill = ['total_actions', 'notes_taken', 'unique_tools_used',
                'section_clicks', 'probe_actions', 'total_gate_crossings',
                'console_open_count', 'unique_consoles']
gate_cols = [c for c in player_profile.columns if c.startswith('gate_')]

for col in cols_to_fill + gate_cols:
    player_profile[col] = player_profile[col].fillna(0)

dur_cols = [c for c in player_profile.columns if c.startswith('dur_')]
player_profile['total_tool_time'] = player_profile[dur_cols].sum(axis=1)
player_profile['notebook_ratio'] = player_profile['dur_notebook'] / player_profile['total_tool_time'].replace(0, np.nan)
player_profile['action_density'] = player_profile['total_actions'] / player_profile['total_tool_time'].replace(0, np.nan)
player_profile['gender_label'] = player_profile['gender'].map({1: 'Male', 2: 'Female'})

print("player_profile shape:", player_profile.shape)
player_profile[['total_actions', 'notes_taken', 'solution_score']].describe().round(2)


---
## 03 — Exploratory Data Analysis

### Why this matters
Before building models, we need to understand the data visually.  
Each chart below answers a specific business question.

### Questions we'll answer
1. Are players generally succeeding? *(score distribution)*
2. Where do players spend their time? *(tool usage)*  
3. Does note-taking actually help? *(notebook vs performance)*
4. Is more activity always better? *(actions vs score)*
5. Which features correlate most with success? *(heatmap)*


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Player Performance Overview', fontsize=15, fontweight='bold')

score_counts = player_profile['solution_score'].value_counts().sort_index()

axes[0].bar(score_counts.index, score_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Score Distribution (0=no solution, 7=perfect)')
axes[0].set_xlabel('Solution Score')
axes[0].set_ylabel('Number of Players')
for score, count in score_counts.items():
    axes[0].text(score, count + 0.3, str(count), ha='center', fontsize=9)

failed = (player_profile['solution_score'] == 0).sum()
struggling = ((player_profile['solution_score'] >= 1) & (player_profile['solution_score'] <= 3)).sum()
successful = (player_profile['solution_score'] >= 4).sum()

sizes = [failed, struggling, successful]
labels = [f'Failed (score=0)\n{failed} players',
          f'Struggling (1-3)\n{struggling} players',
          f'Successful (4-7)\n{successful} players']
colors = ['#e74c3c', '#f39c12', '#2ecc71']

axes[1].pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%', startangle=90)
axes[1].set_title('Player Success Segments')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '01_performance_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print("33% of players scored 0 despite a full 1-hour session.")


In [ ]:
tool_name_map = {
    'dur_alien_db': 'Alien DB',
    'dur_comm_center': 'Comm Center',
    'dur_concepts_db': 'Concepts DB',
    'dur_mission_control': 'Mission Control',
    'dur_missions_db': 'Missions DB',
    'dur_notebook': 'Notebook',
    'dur_periodic_table': 'Periodic Table',
    'dur_probe_design': 'Probe Design',
    'dur_solar_db': 'Solar DB',
    'dur_spectra': 'Spectra'
}

dur_cols = list(tool_name_map.keys())
tool_means = player_profile[dur_cols].mean()
tool_means.index = [tool_name_map[c] for c in tool_means.index]
tool_means = tool_means.sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Tool Usage Analysis', fontsize=15, fontweight='bold')

axes[0].barh(tool_means.index, tool_means.values, color='steelblue', edgecolor='white')
axes[0].set_title('Average Time Spent per Tool (minutes)')
axes[0].set_xlabel('Average Duration (minutes)')
for i, val in enumerate(tool_means.values):
    axes[0].text(val + 0.2, i, f'{val:.1f}', va='center', fontsize=9)

tool_data = player_profile[dur_cols].copy()
tool_data.columns = [tool_name_map[c] for c in tool_data.columns]
tool_order = tool_means.index.tolist()
data_for_box = [tool_data[t].values for t in tool_order]

axes[1].boxplot(data_for_box, labels=tool_order, vert=False, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[1].set_title('Duration Distribution per Tool')
axes[1].set_xlabel('Duration (minutes)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '02_tool_usage.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Notebook Usage vs Performance', fontsize=15, fontweight='bold')

scatter = axes[0].scatter(
    player_profile['notes_taken'],
    player_profile['solution_score'],
    c=player_profile['solution_score'],
    cmap='RdYlGn',
    alpha=0.7,
    s=60,
    edgecolors='white'
)
plt.colorbar(scatter, ax=axes[0], label='Solution Score')

z = np.polyfit(player_profile['notes_taken'], player_profile['solution_score'], 1)
x_vals = np.linspace(player_profile['notes_taken'].min(), player_profile['notes_taken'].max(), 100)
axes[0].plot(x_vals, np.poly1d(z)(x_vals), 'navy', linestyle='--', linewidth=1.5)

corr = player_profile['notes_taken'].corr(player_profile['solution_score'])
axes[0].text(0.05, 0.92, f'r = {corr:.2f}', transform=axes[0].transAxes, fontsize=11, color='navy', fontweight='bold')
axes[0].set_xlabel('Number of Notes Taken')
axes[0].set_ylabel('Solution Score')
axes[0].set_title('Notes Taken vs Solution Score')

no_notes = player_profile[player_profile['notes_taken'] == 0]['solution_score'].mean()
took_notes = player_profile[player_profile['notes_taken'] >= 1]['solution_score'].mean()

axes[1].bar(['No Notes', 'Took Notes'], [no_notes, took_notes],
            color=['#e74c3c', '#2ecc71'], edgecolor='white', width=0.5)
axes[1].set_title('Avg Score: Note Takers vs Non-Note Takers')
axes[1].set_ylabel('Average Solution Score')
axes[1].set_ylim(0, 7)
axes[1].text(0, no_notes + 0.1, f'{no_notes:.2f}', ha='center', fontsize=12, fontweight='bold')
axes[1].text(1, took_notes + 0.1, f'{took_notes:.2f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '03_notebook_vs_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Note-takers avg score: {took_notes:.2f}")
print(f"Non-note-takers avg score: {no_notes:.2f}")


In [ ]:
corr_cols = ['solution_score', 'mc_average', 'total_actions', 'notes_taken',
             'section_clicks', 'probe_actions', 'total_gate_crossings',
             'unique_tools_used', 'total_tool_time', 'notebook_ratio',
             'action_density', 'dur_probe_design', 'dur_notebook', 'dur_alien_db']

corr_matrix = player_profile[corr_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, annot_kws={'size': 8}, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + '05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top correlations with solution_score:")
score_corr = corr_matrix['solution_score'].drop('solution_score').sort_values(key=abs, ascending=False)
print(score_corr.head(6).round(3))


---
## 04 — Player Segmentation (K-Means Clustering)

### Why this matters
Not all players are the same. Treating them as one group leads to one-size-fits-all design decisions  
that work poorly for everyone. Segmentation lets us ask: **who are our players, really?**

### Method: K-Means Clustering
K-Means groups players by minimizing the distance between each player and their cluster center.  
We use **10 behavioral features** (no psychological scores — pure in-game behavior).

**Why StandardScaler first?**  
`total_actions = 750` and `notebook_ratio = 0.07` can't be compared directly.  
Scaling transforms every feature to: *(value − mean) / std deviation*  
Now all features live on the same scale and K-Means distances are meaningful.

**How we chose k=3:**  
- Elbow method: inertia reduction slows around k=3
- Silhouette score: measures cluster separation quality
- Domain knowledge: 3 segments ("good", "medium", "lost") makes business sense


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

segment_colors = {
    'Achievers': '#2ecc71',
    'Explorers': '#3498db',
    'Lost Players': '#e74c3c'
}
seg_order = ['Lost Players', 'Explorers', 'Achievers']

cluster_features = [
    'total_actions', 'notes_taken', 'section_clicks', 'probe_actions',
    'total_gate_crossings', 'unique_tools_used', 'dur_notebook',
    'dur_probe_design', 'notebook_ratio', 'action_density'
]

cluster_data = player_profile[cluster_features].dropna()
valid_idx = cluster_data.index

scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_data)

inertias = []
sil_scores = []

for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Finding Optimal Number of Segments', fontsize=14, fontweight='bold')

axes[0].plot(range(2, 8), inertias, 'o-', color='steelblue', linewidth=2)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(range(2, 8), sil_scores, 's-', color='coral', linewidth=2)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score (higher = better)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '06_optimal_k.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

player_profile['segment'] = np.nan
player_profile.loc[valid_idx, 'segment'] = labels
player_profile['segment'] = player_profile['segment'].astype('Int64')

seg_scores = player_profile.groupby('segment')['solution_score'].mean()
score_rank = seg_scores.rank()

segment_labels = {}
for seg, rank in score_rank.items():
    if rank == 3:
        segment_labels[seg] = 'Achievers'
    elif rank == 2:
        segment_labels[seg] = 'Explorers'
    else:
        segment_labels[seg] = 'Lost Players'

player_profile['segment_name'] = player_profile['segment'].map(segment_labels)

summary = player_profile.groupby('segment_name').agg(
    count=('user_id', 'count'),
    avg_score=('solution_score', 'mean'),
    avg_actions=('total_actions', 'mean'),
    avg_notes=('notes_taken', 'mean')
).round(2)

print(summary.reindex(seg_order))


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Player Segmentation Results', fontsize=15, fontweight='bold')

for seg_name in seg_order:
    mask = player_profile.loc[valid_idx, 'segment_name'] == seg_name
    color = segment_colors[seg_name]
    count = mask.sum()
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, label=f'{seg_name} (n={count})',
                    alpha=0.75, s=70, edgecolors='white')

axes[0].set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')
axes[0].set_title('Player Segments (PCA)')
axes[0].legend()

seg_means = player_profile.groupby('segment_name')['solution_score'].mean().reindex(seg_order)
seg_stds = player_profile.groupby('segment_name')['solution_score'].std().reindex(seg_order)
bar_colors = [segment_colors[s] for s in seg_order]

axes[1].bar(seg_order, seg_means.values, color=bar_colors, edgecolor='white',
            width=0.5, yerr=seg_stds.values, capsize=6)
axes[1].set_title('Average Solution Score by Segment')
axes[1].set_ylabel('Average Solution Score (0-7)')
axes[1].set_ylim(0, 8)
for i, (seg, val) in enumerate(seg_means.items()):
    count = (player_profile['segment_name'] == seg).sum()
    axes[1].text(i, val + seg_stds[seg] + 0.2, f'{val:.1f}\n(n={count})',
                 ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '07_segments_overview.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
radar_features = ['total_actions', 'notes_taken', 'section_clicks',
                  'probe_actions', 'total_gate_crossings', 'notebook_ratio']
radar_labels = ['Total\nActions', 'Notes\nTaken', 'Section\nClicks',
                'Probe\nActions', 'Gate\nCrossings', 'Notebook\nRatio']

seg_radar = player_profile.groupby('segment_name')[radar_features].mean()

mm = MinMaxScaler()
seg_radar_norm = pd.DataFrame(
    mm.fit_transform(seg_radar),
    index=seg_radar.index,
    columns=radar_features
)

N = len(radar_features)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for seg_name in seg_order:
    if seg_name not in seg_radar_norm.index:
        continue
    values = seg_radar_norm.loc[seg_name].tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2.5, label=seg_name, color=segment_colors[seg_name])
    ax.fill(angles, values, alpha=0.12, color=segment_colors[seg_name])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, size=11)
ax.set_ylim(0, 1)
ax.set_title('Behavioral Fingerprint by Segment', size=14, fontweight='bold', pad=25)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + '08_segment_radar.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 05 — Early Warning System

### Why this matters
If we can identify struggling players **before they fail**, we can intervene:  
show a hint, redirect them to a key tool, or trigger a tutorial prompt.

### Approach
1. Isolate each player's **first 20 minutes** of gameplay
2. Extract 7 behavioral signals from that window
3. Test whether those signals predict final score (Random Forest)
4. Build a **risk score** as an actionable output

### Key result (spoiler)
The first 20 minutes **cannot reliably predict** final score alone (R² ≈ -0.07).  
This is itself an important finding — the critical decision point happens **mid-game**, not at the start.  
However, a composite risk score based on 4 early signals still meaningfully separates player groups.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

WINDOW = 20

first_events = log_raw.groupby('user_id')['timestamp'].min()
first_events.name = 'session_start'

log_timed = log_raw.merge(first_events, on='user_id')
log_timed['minutes_elapsed'] = (log_timed['timestamp'] - log_timed['session_start']).dt.total_seconds() / 60

early_log = log_timed[log_timed['minutes_elapsed'] <= WINDOW].copy()

print(f"Total events: {len(log_timed)}")
print(f"Events in first {WINDOW} min: {len(early_log)} ({len(early_log)/len(log_timed)*100:.1f}%)")


In [ ]:
early_stats = early_log.groupby('user_id').agg(
    early_actions=('action', 'count'),
    early_unique_tools=('tool', 'nunique'),
    early_section_clicks=('action', lambda x: (x == 'Click Section').sum())
).reset_index()

note_rows_early = early_log[early_log['action'].str.contains('Creat Note', na=False)]
early_notes = note_rows_early.groupby('user_id')['action'].count()
early_notes.name = 'early_notes'

probe_rows_early = early_log[early_log['tool'].str.contains('Probe|probe', na=False)]
early_probe = probe_rows_early.groupby('user_id')['action'].count()
early_probe.name = 'early_probe'

early_stats = early_stats.merge(early_notes, on='user_id', how='left')
early_stats = early_stats.merge(early_probe, on='user_id', how='left')
early_stats['early_notes'] = early_stats['early_notes'].fillna(0)
early_stats['early_probe'] = early_stats['early_probe'].fillna(0)

early_stats['tried_probe_early'] = (early_stats['early_probe'] > 0).astype(int)
early_stats['took_notes_early'] = (early_stats['early_notes'] > 0).astype(int)

early_analysis = early_stats.merge(
    player_profile[['user_id', 'solution_score', 'segment_name']],
    on='user_id',
    how='inner'
)

print("Early analysis shape:", early_analysis.shape)
early_analysis.head()


In [ ]:
feat_cols = ['early_actions', 'early_unique_tools', 'early_notes',
             'early_probe', 'early_section_clicks', 'tried_probe_early', 'took_notes_early']

rf = RandomForestRegressor(n_estimators=100, random_state=42)
cv_scores = cross_val_score(rf, early_analysis[feat_cols], early_analysis['solution_score'], cv=5, scoring='r2')

print(f"R² score (5-fold CV): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print("The first 20 minutes alone cannot predict final score reliably.")
print("The critical engagement point is mid-game, not the opening.")

rf.fit(early_analysis[feat_cols], early_analysis['solution_score'])
importances = pd.Series(rf.feature_importances_, index=feat_cols).sort_values(ascending=True)

low_action_threshold = early_analysis['early_actions'].quantile(0.25)
low_tool_threshold = early_analysis['early_unique_tools'].quantile(0.25)

early_analysis['risk_low_activity'] = (early_analysis['early_actions'] < low_action_threshold).astype(int)
early_analysis['risk_no_notes'] = (early_analysis['took_notes_early'] == 0).astype(int)
early_analysis['risk_no_probe'] = (early_analysis['tried_probe_early'] == 0).astype(int)
early_analysis['risk_low_tools'] = (early_analysis['early_unique_tools'] < low_tool_threshold).astype(int)

risk_factors = ['risk_low_activity', 'risk_no_notes', 'risk_no_probe', 'risk_low_tools']
early_analysis['risk_score'] = early_analysis[risk_factors].sum(axis=1) / 4 * 100

early_analysis['risk_group'] = pd.cut(
    early_analysis['risk_score'],
    bins=[-1, 25, 50, 75, 101],
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical Risk']
)

risk_summary = early_analysis.groupby('risk_group', observed=True).agg(
    count=('user_id', 'count'),
    avg_score=('solution_score', 'mean')
).round(2)

print(risk_summary)


In [ ]:
risk_order = ['Low Risk', 'Medium Risk', 'High Risk', 'Critical Risk']
risk_colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']

risk_counts = early_analysis['risk_group'].value_counts().reindex(risk_order)
risk_perf = early_analysis.groupby('risk_group', observed=True)['solution_score'].mean().reindex(risk_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Player Risk Score — Early Warning System', fontsize=14, fontweight='bold')

axes[0].bar(risk_order, risk_counts.values, color=risk_colors, edgecolor='white', width=0.5)
axes[0].set_title('Risk Group Distribution')
axes[0].set_ylabel('Number of Players')
for i, val in enumerate(risk_counts.values):
    axes[0].text(i, val + 0.3, str(val), ha='center', fontsize=11, fontweight='bold')

axes[1].bar(risk_order, risk_perf.values, color=risk_colors, edgecolor='white', width=0.5)
axes[1].set_title('Avg Final Score by Risk Group')
axes[1].set_ylabel('Average Solution Score (0-7)')
axes[1].set_ylim(0, 7)
for i, val in enumerate(risk_perf.values):
    axes[1].text(i, val + 0.1, f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '11_risk_scores.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 06 — Tool Analysis & Design Recommendations

### Why this matters
The game has 10 tools. Not all of them are equal:
- Some are used by almost everyone but don't help (wasted screen real estate)
- Some are rarely discovered but strongly predict success (hidden gems)
- Some need to be redesigned entirely

### Tool Positioning Matrix
We evaluate each tool on two axes:
- **X axis:** Adoption rate — what % of players used it for ≥1 minute?
- **Y axis:** Value — correlation with solution score
- **Bubble size:** Average time spent

This creates 4 quadrants: Core Tools · Hidden Gems · Busy Tools · Dead Weight


In [ ]:
tool_name_map2 = {
    'dur_alien_db': 'Alien DB',
    'dur_comm_center': 'Comm Center',
    'dur_concepts_db': 'Concepts DB',
    'dur_mission_control': 'Mission Control',
    'dur_missions_db': 'Missions DB',
    'dur_notebook': 'Notebook',
    'dur_periodic_table': 'Periodic Table',
    'dur_probe_design': 'Probe Design',
    'dur_solar_db': 'Solar DB',
    'dur_spectra': 'Spectra'
}

tool_stats = {}
for col, name in tool_name_map2.items():
    avg_time = player_profile[col].mean()
    corr = player_profile[col].corr(player_profile['solution_score'])
    adoption = (player_profile[col] >= 1).sum() / len(player_profile) * 100
    tool_stats[name] = {'avg_time': avg_time, 'corr_with_score': corr, 'adoption_rate': adoption}

tool_stats_df = pd.DataFrame(tool_stats).T.round(3)
print(tool_stats_df.sort_values('corr_with_score', ascending=False))


In [ ]:
mid_x = tool_stats_df['adoption_rate'].mean()
colors_tool = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(11, 8))

for i, (tool_name, row) in enumerate(tool_stats_df.iterrows()):
    bubble_size = row['avg_time'] * 25 + 80
    ax.scatter(row['adoption_rate'], row['corr_with_score'],
               s=bubble_size, color=colors_tool[i],
               alpha=0.8, edgecolors='white', linewidth=1.5)
    ax.annotate(tool_name,
                xy=(row['adoption_rate'], row['corr_with_score']),
                xytext=(8, 4), textcoords='offset points',
                fontsize=9, fontweight='bold')

ax.axvline(x=mid_x, color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

ymax = tool_stats_df['corr_with_score'].max()
ymin = tool_stats_df['corr_with_score'].min()
xmin = tool_stats_df['adoption_rate'].min()

ax.text(mid_x + 0.3, ymax * 0.85, 'CORE TOOLS
(keep & enhance)', fontsize=9, color='green', fontweight='bold')
ax.text(xmin, ymax * 0.85, 'HIDDEN GEMS
(promote early)', fontsize=9, color='steelblue', fontweight='bold')
ax.text(mid_x + 0.3, ymin * 0.85, 'BUSY TOOLS
(simplify)', fontsize=9, color='orange', fontweight='bold')
ax.text(xmin, ymin * 0.85, 'DEAD WEIGHT
(redesign)', fontsize=9, color='red', fontweight='bold')

ax.set_xlabel('Adoption Rate (% players who used >= 1 min)', fontsize=11)
ax.set_ylabel('Correlation with Solution Score', fontsize=11)
ax.set_title('Tool Positioning Matrix
(bubble size = avg time spent)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '12_tool_positioning_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
print("TOOL RECOMMENDATIONS")
print("-" * 60)

for tool_name, row in tool_stats_df.sort_values('corr_with_score', ascending=False).iterrows():
    a = row['adoption_rate']
    c = row['corr_with_score']

    if a >= mid_x and c >= 0:
        tag = "CORE TOOL    → Keep & surface early"
    elif a < mid_x and c >= 0:
        tag = "HIDDEN GEM   → Add to onboarding"
    elif a >= mid_x and c < 0:
        tag = "BUSY TOOL    → Simplify content"
    else:
        tag = "DEAD WEIGHT  → Redesign"

    print(f"{tool_name:<18} | adoption: {a:.1f}% | corr: {c:+.3f} | {tag}")


---
## 07 — Conclusions & Key Takeaways

### What the data tells us

| # | Finding | Business Implication |
|---|---|---|
| 1 | **33% of players score 0** despite a full session | Onboarding and tool discoverability must be redesigned |
| 2 | **Note-taking is the strongest behavioral predictor** | Notebook must become a first-class mechanic, not hidden |
| 3 | **Lost Players are busiest but least strategic** | A hint system triggered by unfocused behavior can help |
| 4 | **First 20 minutes don't predict final score** | Critical engagement window is mid-game |
| 5 | **Missions DB has negative score correlation** | Review content — players may be spending time unproductively |
| 6 | **Probe Design is underused despite high value** | Surface it in the first 5 minutes of gameplay |

---

### Limitations
- Sample size is n=159 — findings are exploratory, not statistically definitive
- All players are undergraduate students from one university — limited generalizability
- Correlation ≠ causation — further A/B testing needed to validate design recommendations

---

### Next Steps
1. **A/B test** notebook tutorial vs no tutorial → measure score impact
2. **Implement risk score** in real-time game telemetry → trigger hints at the 20-min mark
3. **Restructure Missions DB** content based on what high-scorers actually use it for
4. **Expand dataset** with more diverse player populations

---

*Dataset: Liu & Liu (2019) — University of Texas at Austin, Alien Rescue Research Team*  
*Analysis: Python · pandas · scikit-learn · matplotlib · seaborn*
